
# Inverse FORM with a cantilever beam


This example uses the cantilever beam model from the OpenTURNS use cases
to demonstrate the :class:`~otrobopt.InverseFORM` algorithm.

The beam tip displacement is given by:

\begin{align}d = \frac{F L^3}{3 E I}\end{align}

where $E$ is Young's modulus, $F$ is the load,
$L$ is the length, and $I$ is the moment of inertia.

We treat the length $L$ as a parameter to calibrate. Given a
target reliability index $\beta_t$, we seek the beam length
such that the Hasofer-Lind reliability index of the failure event
$\{d > d_0\}$ (displacement exceeding a threshold) equals
$\beta_t$.



In [ ]:
import openturns as ot
import otrobopt
from openturns.usecases import cantilever_beam

ot.RandomGenerator.SetSeed(0)

# Load the cantilever beam use case
cb = cantilever_beam.CantileverBeam()
print(cb.model)

Freeze the beam length L (input index 2) as a parameter with initial value 2.55 m.
The remaining random inputs are E, F, I.



In [ ]:
g = ot.ParametricFunction(cb.model, [2], [2.55])

# Build the joint distribution of the random inputs (E, F, I)
marginals = [cb.E, cb.F, cb.II]
distribution = ot.JointDistribution(marginals)

# Starting point: median values of the random inputs
x0 = [dist.computeQuantile(0.5)[0] for dist in marginals]

Define the failure event: displacement > 0.15 m.



In [ ]:
vect = ot.RandomVector(distribution)
output = ot.CompositeRandomVector(g, vect)
event = ot.ThresholdEvent(output, ot.Greater(), 0.15)

Run the inverse FORM algorithm with a target reliability index
$\beta_t = 3.0$.



In [ ]:
algo = otrobopt.InverseFORM(event, 'L', x0)
algo.setTargetBeta(3.0)
algo.run()

result = algo.getResult()
print('Calibrated L =', result.getParameter())
print('Hasofer-Lind beta =', result.getHasoferReliabilityIndex())
print('Convergence criteria =', result.getConvergenceCriteria())

The calibrated length $L^*$ is the beam length that achieves
the target reliability index $\beta_t = 3.0$ for the given
failure threshold. If the initial length leads to a reliability
above the target, the algorithm increases it (making the beam more
flexible and thus less reliable), and vice versa.

